In [1]:
import json
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

project_root = Path.cwd().parent
os.chdir(project_root)

from src.rag_pipeline import RagPipeline
from src.generator import QwenGenerator

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load models
rag = RagPipeline()
generator = QwenGenerator()

# Load dataset
with open("data/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(f"Loaded {len(qa_data)} Q&A pairs")

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Loaded 172 Q&A pairs


In [ ]:
import random


random.seed(42)
eval_samples = random.sample(qa_data, min(10, len(qa_data)))

results = []

for sample in tqdm(eval_samples, desc="Generating answers"):
    question = sample["question"]
    reference = sample["answer"]

    retrieved = rag.retrieve(question, top_k=3)
    context_rag = "\n".join(
        f"[{i}] Q: {r['question']}\nA: {r['answer']}"
        for i, r in enumerate(retrieved, 1)
    )[:2500]

    # Generate with RAG
    answer_rag = generator.generate(question, context=context_rag, max_tokens=128)

    # Generate without RAG (baseline)
    answer_baseline = generator.generate(question, context="", max_tokens=128)

    results.append(
        {
            "question": question,
            "reference": reference,
            "answer_rag": answer_rag,
            "answer_baseline": answer_baseline,
            "sources": [r["question"] for r in retrieved],
        }
    )

print(f"Generated {len(results)} answer pairs")

Generating answers: 100%|██████████| 10/10 [01:12<00:00,  7.28s/it]

Generated 10 answer pairs


In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import evaluate


rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def compute_metrics(reference, generated):
    # ROUGE
    rouge_scores = rouge.score(reference, generated)

    # BLEU
    ref_tokens = reference.split()
    gen_tokens = generated.split()
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie)

    return {
        "rouge1": rouge_scores["rouge1"].fmeasure,
        "rouge2": rouge_scores["rouge2"].fmeasure,
        "rougeL": rouge_scores["rougeL"].fmeasure,
        "bleu": bleu,
    }

In [11]:
metrics_rag = []
metrics_baseline = []

for result in tqdm(results, desc="Computing metrics"):
    metrics_rag.append(compute_metrics(result["reference"], result["answer_rag"]))
    metrics_baseline.append(
        compute_metrics(result["reference"], result["answer_baseline"])
    )

Computing metrics: 100%|██████████| 10/10 [00:00<00:00, 136.97it/s]


In [ ]:
import numpy as np


def average_metrics(metrics_list):
    return {
        metric: np.mean([m[metric] for m in metrics_list])
        for metric in metrics_list[0].keys()
    }


avg_rag = average_metrics(metrics_rag)
avg_baseline = average_metrics(metrics_baseline)

comparison = pd.DataFrame({"RAG": avg_rag, "Baseline": avg_baseline}).T


comparison.round(4)

,rouge1,rouge2,rougeL,bleu
RAG,0.4914,0.3240,0.3733,0.1466
Baseline,0.3189,0.0877,0.1773,0.0412


In [20]:
eval_results = {
    "samples": results,
    "metrics_rag": metrics_rag,
    "metrics_baseline": metrics_baseline,
    "average_rag": avg_rag,
    "average_baseline": avg_baseline,
}

with open("data/evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)